# Lecture 23 — Real Files Fight Back: Reading and Combining Messy Data
### EES 3350/5350 · Python in Earth Science · Friday, October 30, & Monday, November 2n 2026 

Almost every file we've read so far was polite: commas between columns, one header row, dots for decimals. Real data files aren't always like that. Today's files come from a glacier in Norway, and they push back in about five different ways.

We'll cover:
1. **Read, check, fix, repeat**: getting a stubborn file to laod **correctly** (Friday)
2. **Combining pieces:** gluing two files into one table tiwh `np.concatenate` and `pd.concat` (Monday)
3. **Adding it up:** running totals with `.cumsum()` (Monday)
4. **Savingj a summary:** a small results table you can share (Monday)

**Learning objectives**
- look at a raw file **before** reading it
- use `encoding=`, `sep=`, `decimal=`, `skiprows=`, `nrows=`, `header=` + `names=`, and `na_values=`
- check `.dtypes` after every read, because a file that loads with the wrong types is more dangerous than one that throws an error
- stack arrays with `np.concatenate` and tables with `pd.concat`, and explain why `pd.concat` is safer
- clean up a combined table with `ignore_index=`, `.drop_duplicates()`, and `.sort_values()`
- compute a running total with `.cumsum()`
- build a small DataFrame from a dictionary and save it with `.to_csv()`

---
## Part 0 : setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

data = "../../course-materials/week10/"

## Meet the glacier

**Rembesdalskåka** is an outlet glacier of **Hardangerjøkulen**, an ice cap in southern Norway. The Norwegian Water Resources and Energy Directorate (**NVE**) has measured its **mass balance** every year since 1963: how much snow it gained and how much ice it lost.

|**column**|**name**|**what it means**|**sign**|
|:--|:--|:--|:--|
|`Bw`|winter balance|snow added over the winter|positive|
|`Bs`|summer balance|snow and ice melted over the summer|negative|
|`Ba`|annual balance|the year's net change, `Bw + Bs`|either|

All three are in **meters of water equivalent (m w.e.)**: if you melted the snow or ice that was gained (or lost) and spread the water evenly over the whole glacier, this is how deep it would be. A positive `Ba` year means the glacier **gained** mass.

<span style="font-size:0.85em;color:#6b7a8d;"><em>Data: NVE, 2026. Norwegian Water Resources and Energy Directorate (NVE). Climate indicator products, http://glacier.nve.no/viewer/CI/, downloaded 2026.10.28. Kjøllmoen, B. (Ed.), L.M. Andreassen, H. Elvehøy (2025). Glaciological investigations in Norway 2024. NVE Rapport 27-2025.</em></span>

---
## Part 1 : read, check, fix, repeate

### Step 0 : look at the file first

Before you read a file into Python, look at it. Remember the `!` runs a terminal command from a noteook cell (like `!pwd`) and `{data}` drops in our Python variable. `head -3` shows the first 3 lines. We've done this before.

In [ ]:
!head -3 {data}rembesdalskaka_massbalance_nve.csv

#### Okay what do we see:
1. **semicolons** (`;`) separate the columns, not commas
2. Numbers like `17,569` and `1,16` use a **comma as the decimal point**, the way most of Europe writes numbers. So `1,16` means 1.16.
3. Some characters are **garbled** (look at the names in the reference column). Norwegian has letters like ø and å, and this is a clue that file stores text in an unusual way.

Let's try reading it in the normal way:

In [ ]:
nve = pd.read_csv(data + "rembesdalskaka_massbalance_nve.csv")

### That didn't work. Let's work on fixing it.

#### Fix 1: `encoding=`
A file is really just a long list of **bytes** (numbers from 0 to 255). An **encoding** is the rule for turning bytes into characters.

Pandas assumes **UTF-8**, today's standard. THis file was saved in an older Western European encoding called **Latin-1**, where the byte `0xf8` means **ø** (and I had to copy and paste that letter in this Markdown cell). In UTF-8, that byte isn't allowed in that spot, so pandas gives up.

You can read the error message from the bottom: *can't decode byte 0xf8 in position 246*. The 246th byte of the file is the first ø.

<div class="alert alert-success">

**Tip: how do I know which encoding a file uses?** Ask the terminal: `!file -i path` prints a line ending in `charset=...`. Here it says `iso-8859-1`, which is another name for Latin-1. If it says `utf-8` or `us-ascii`, the default works. You can't always be sure, so the habit is: try the default, switch to `encoding="latin-1"` if you get a `UnicodeDecodeError`, then **look** at the text. Latin-1 never raises an error, so letters like ø showing up as `�` or `Ã¸` mean you still have the wrong encoding.

</div>

In [ ]:
!file -i {data}rembesdalskaka_massbalance_nve.csv

##### Let's try again pointing to the right encoding:

In [ ]:
nve = pd.read_csv(data + "rembesdalskaka_massbalance_nve.csv", encoding="latin-1")

### A **different** error! This means progress. 
*Expected 7 fields in line 3, saw 12*: pandas is splitting each line at **commas**, but in this file the commas are decimal points, so every row splits into a different number of pieces.

#### Fix 2 : `sep=`
`sep=` tells pandas which character separates the columns. This is also **why** the file uses semicolons: if the comma is your decimal point, it can't also separate your columns.

In [ ]:
nve = pd.read_csv(data + "rembesdalskaka_massbalance_nve.csv", encoding="latin-1", sep=";")

nve.head(3)

### No error! Are we done?
**Check the types**:

In [ ]:
nve[["Year", "Area", "Bw", "Bs", "Ba"]].dtypes

`Year` is a number, but `Area`, `Bw`, `Bs`, and `Ba` are `object`: **text**. Here's what happens if you don't notice that:

In [ ]:
(nve["Bw"] + nve["Bs"]).head(3)

There's no error, but this also doesn't make sense. We tried to do addition (`+`). Because these data are text, the `+` **glues** the strings together so `"1,16" + "-2,58"` became `"1,16-2,58"`.

<center><b>This is (in my opinion) the worst kind of bug: everything runs, and nothing is right.</b></center>

#### Fix 3 : `decimal=`
`decimal=","` tells pandas that a comma inside a number is the decimal point. Let's redo these operations, but including `decimal=` in our `read_csv` call:

In [ ]:
nve = pd.read_csv(data + "rembesdalskaka_massbalance_nve.csv", encoding="latin-1", sep=";", decimal=",")

nve[["Year", "Area", "Bw", "Bs", "Ba"]].dtypes
(nve["Bw"] + nve["Bs"]).head(3)

#### omg real numbers!

Okay. Now we have one more check to do:
- `Ba` should equal `Bw + Bs`. Does it?

In [ ]:
difference = nve["Bw"] + nve["Bs"] - nve["Ba"]

print(f"biggest difference: {difference.abs().max():.2f} m w.e.")

In 53 years, `Bw + Bs` equals `Ba` exactly. In 8 years it's off by 0.01, and in one year (1987) by 0.02. How close is close enough?

Each number in the file is rounded to 2 decimals, so each one can be off by up to 0.005. Our check adds and subtracts **three** rounded numbers, so rounding alone can make the answer off by up to 3 × 0.005 = 0.015. That explains every year except 1987, which is off by a little more, possibly because NVE re-analyzed the older years and adjusted one of the values.

**Is 0.02 m w.e. a problem?** A typical winter balance here is about 2 m w.e., so it's a 1% difference in one year out of 62. If the file had been read wrong (say, two columns swapped), the differences would be **meters**, not hundredths. 
<center><b>The file passes.</b></center>

<div class="alert alert-success">

**Tip: read → check → fix → repeat.** After every `pd.read_csv`, look at `.head()` **and** `.dtypes`. Fix one thing, then read again. An error message is the *good* outcome, because it tells you something is wrong. A file that loads with the wrong types is the dangerous one.

</div>

### Generally...

|**the file has...**|**symptom**|**fix**|
|:--|:--|:--|
|a non-UTF-8 encoding|`UnicodeDecodeError`|`encoding="latin-1"`|
|a separator that isn't a comma|`ParserError`, or everything in one column|`sep=";"` (or `"\t"` for tabs)|
|commas as decimal points|number columns are `object` (text)|`decimal=","`|


### Now a first look at 62 years of mass balance. 
Lecture 21's bar chart, with each bar colored by its sign:

In [ ]:
colors = np.where(nve["Ba"] > 0, "tab:blue", "tab:red")

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.bar(nve["Year"], nve["Ba"], color=colors)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_xlabel("year")
ax.set_ylabel("annual balance (m w.e.)")
ax.set_title("Rembesdalskåka annual mass balance (blue = gained, red = lost)");

<div class="alert alert-warning">

**Question:** 29 of the 62 years are blue. From this chart, **is the glacier growing or shrinking?** Is it easy to tell? (We'll answer this properly on Monday.)

</div>

> your answer here

<div class="alert alert-info">

### Exercise 1 : a file with two header rows

`hardangerjokulen_bands_2006-2007.csv` splits the ice cap into **elevation bands** and gives the winter and summer balance for each band in a single year. (We don't know who first published this table, so we'll only use it to practice reading.)

1. Look at the raw file with `!head`. Identify which rows need to be skipped, if any?
2. Read it so that **every column is a number**. Check `.dtypes` to prove it.
3. The column names are long, have spaces in them, and are annoying to type. Use `.rename()` (Lecture 18) to give them short names: `elev_min`, `elev_max`, `area_km2`, `winter`, and `summer`.
4. Add a column `net` = `winter` + `summer`  for each band.
5. Which bands **gained** mass (`net > 0`)? Between which two elevations does `net` change from positive to negative?

</div>

<details>
<summary style="color:green;"><strong>Hint 1</strong></summary>

What's on the **second** line of the file? Both header lines become part of the table if you don't skip one. Remember `skip=` and indexing?

In [ ]:
# your answer here
# 1
!head -3 {data}hardangerjokulen_bands_2006-2007.csv

In [ ]:
bands = pd.read_csv(data + "hardangerjokulen_bands_2006-2007.csv", skiprows=[____])

In [ ]:
# 2
print(bands.dtypes)

<details>
<summary style="color:green;"><strong>Hint 2</strong></summary>

For step 3, `.rename()` takes a dictionary of `{"old name": "new name"}`. To get the old names exactly right, run `print(list(bands.columns))` and copy them. If a rename seems to do nothing, or you get a `KeyError` for a column you can *see*, look **very** closely at the end of each name.

</details>

In [ ]:
# 3
print(list(bands.columns))

In [ ]:
bands = bands.rename(columns={___________: "elev_min",
                            ___________: "elev_max",
                            ___________: "area_km2",
                            ___________: "winter",
                            ___________: "summer"})

In [ ]:
# 4
bands["net"] = _______________ + _________________

In [ ]:
# 5
print(bands[________________])

<div class="alert alert-danger">

**Common Error:** a `KeyError` for a column name you can see. Column names can hide spaces at the start or end: `"Specific summer "` is not `"Specific summer"`. Print `list(df.columns)` to see them, then fix one name with `.rename()`, or strip spaces from every name at once with `df.columns = df.columns.str.strip()`.

</div>

---
## Part 2 : combining pieces

Real data often arrives in pieces: different years, different field crews, different instruments. To practice, **we split NVE's data into two pieces on purpose**:
- one for 1963-1999
- one for 1998-2024

The second piece came with a short README. 

<div class="alert alert-success">

**What's a README?** A short plain-text note that travels with data or code, written so that a stranger can use it without asking you questions: what it is, how it's organized, what's tricky about it, and how to cite it. The name is an instruction: *read me first*. Without this one, nothing in the file would tell you that summer comes before winter. On GitHub, a repository's `README.md` is shown on its front page, and you'll write your own for Lab 10 and your final project.

</div>

#### Let's look at all three:

In [ ]:
!head -3 {data}rembesdalskaka_balance_1963-1999.csv
!echo ---
!head -3 {data}rembesdalskaka_balance_1998-2024.csv
!echo ---
!cat {data}rembesdalskaka_balance_1998-2024_README.txt

The second file has **no header row**, its columns are in a **different** order (summer before winter), and the two files **overlap** in 1998 and 1999.

### NumPy first: `np.concatenate`
You know `np.loadtxt` from Lecture 14. Load both files as arrays:

In [ ]:
early_arr = np.loadtxt(data + "rembesdalskaka_balance_1963-1999.csv", delimiter=",", skiprows=1)
late_arr = np.loadtxt(data + "rembesdalskaka_balance_1998-2024.csv", delimiter=",")

print(early_arr.shape, late_arr.shape)

`np.concatenate` glues arrays together along one axis. The `axis` rule from Lecture 16 still applies: the axis number is a position in `.shape`. `axis=0` stacks the **rows**, so the first number in the shape grows and the second one has to match:

|**array**|**shape**|
|:--|:--|
|`early_arr`|(37, 4)|
|`late_arr`|(27, 4)|
|`np.concatenate([early_arr, late_arr], axis=0)`|(37 + 27, 4) = **(64, 4)**|

In [ ]:
stacked = np.concatenate([early_arr, late_arr], axis=0)

print(stacked.shape)

#### It worked...didn't it?
Column 1 is supposed to be the winter balance. Plot it:

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(stacked[:, 0], stacked[:, 1], marker="o")
ax.axhline(0, color="black", linewidth=0.8)
ax.set_xlabel("year")
ax.set_ylabel("column 1 (m w.e.)")
ax.set_title("'Winter balance' from np.concatenate");

#### Yikes.
Winter balance is the snow a glacier **gains**, so it should always be positive (on this glacier, it has never been below 0.8 m w.e.). After 1998, the values suddenly turn negative. They are **summer** values: an array has no column names, so `np.concatenate` stacked the second file's summer column right under the first file's winter column. The shapes matched, so there was no error.

<div class="alert alert-danger">

**Common Error:** `np.concatenate` only checks that the **shapes** fit. It has no idea what the columns *mean*. If two files have their columns in a different order, NumPy will mix them up silently.

</div>

<div class="alert alert-warning">

**Question:** How could you fix `late_arr` before stacking it, using indexing you already know?

</div>

<details>
<summary style="color:green;"><strong>Answer</strong></summary>

Reorder its columns with a list of column positions: `late_arr[:, [0, 2, 1, 3]]` takes column 0 (year), then 2 (winter), then 1 (summer), then 3 (annual). That works, but **you** have to know the order and remember to do it, every time.

</details>

#### Luckily, we have pandas...

### pandas: `pd.concat` lines up columns by name

Now let's read both files as DataFrames:

In [ ]:
early = pd.read_csv(data + "rembesdalskaka_balance_1963-1999.csv")
late = pd.read_csv(data + "rembesdalskaka_balance_1998-2024.csv")

print(early.shape, late.shape)
late.head(3)

`late` has **26** rows, not 27, and its column names are numbers. With no header row, pandas used the first line of **data** (the year 1998) as the header, so a whole year of emasurements silently disappeared.

`header=none` says "there's no header row", and `names=` gives the column names yourself, **in the order the README** lists tehm:

In [ ]:
late = pd.read_csv(data + "rembesdalskaka_balance_1998-2024.csv", header=None, names=["Year", "Bs", "Bw", "Ba"])

print(late.shape)
late.head(3)

Now `pd.concat` takes a **list** of DataFrames and stacks them. Say the newer file happened to come first in our list (that happens when you read every file in a folder, as you'll see later):

In [ ]:
both = pd.concat([late, early])

print(both.shape)
both.head(3)

pandas matched the columns **by name**, so `Bw` is still the winter balance in every row, even though the two files listed their columns in a different order.

#### Three problems are left:


In [ ]:
print(both.loc[0])                                # 1. the index labels repeat
print(both["Year"].duplicated().sum(), "repeated years")   # 2. 1998 and 1999 are in twice
print(both["Year"].head(3).tolist())               # 3. the years are out of order

1. **the index labels repeat**
    - each piece kept its own index (0, 1, 2, 3...), so `.loc[0]` now finds **two** rows.
    - `ignore_index=True` numbers the combined rows 0, 1, 2, ... from scratch.
2. **1998 and 1999 are in twice**
    - `.drop_duplicates()` removes any row hat is an exact copy of an earlier row.
3. **the years are out of order**
    - `.sort_values("Year")` puts the rows in order of a column.
   

In [ ]:
both = pd.concat([late, early], ignore_index=True)
print(len(both), "rows after concat")

both = both.drop_duplicates()
print(len(both), "rows after drop_duplicates")

both = both.sort_values("Year", ignore_index=True)
both.head()

#### That's much better.
Whenever you rebuild a dataset, **check it against something you trust.** We have the original NVE file, so the rebuilt annual balances should match it exactly.

In [ ]:
print(len(both), len(nve))
print((both["Ba"].to_numpy() == nve["Ba"].to_numpy()).all())

<div class="alert alert-danger">

**Common Error:** `.drop_duplicates()` only removes rows where **every** column matches. If an overlap year was revised in the newer file (say, the 1999 winter balance changed from 1.82 to 1.85), **both** versions stay, and you'll have two different 1999s. After combining, check `df["Year"].duplicated().sum()` is 0.

</div>

|**you want**|**NumPy**|**pandas**|
|:--|:--|:--|
|stack pieces on top of each other|`np.concatenate([a, b], axis=0)`|`pd.concat([df1, df2])`|
|columns matched by...|position only|**name**|
|a fresh 0, 1, 2, ... index|(always)|`ignore_index=True`|
|a file with no header row|(no headers anyway)|`header=None, names=[...]`|

<div class="alert alert-info">

### Exercise 2 : check the combined table

1. On one set of axes, plot `Bw` and `Bs` from `both` against `Year`, with a legend. Confirm that the winter balance is now positive in every year.
2. Which year had the **most snow** (largest `Bw`)? Which had the **most melt** (most negative `Bs`)? Print the year and the value for each.

</div>

<details>
<summary style="color:green;"><strong>Hint</strong></summary>

`.idxmax()` gives the **index label** of the biggest value. Because `both` has a plain 0, 1, 2, ... index, use that label with `.loc` to get the matching `Year`.

</details>

In [ ]:
# your code here
# 1
fig, ax = plt.subplots(figsize=(9, 3.5))

ax.plot(____________, ____________, marker="o", label="winter balance")
ax.plot(____________, ____________, marker="o", label="summer balance")
ax.axhline(0, color="black", linewidth=0.8)

ax.set_xlabel("year")
ax.set_ylabel("balance (m w.e.)")
ax.legend();

# 2
snowy = ____________
melty = ____________
print(f"most snow: {both.loc[snowy, 'Year']} ({both.loc[snowy, 'Bw']} m w.e.)")
print(f"most melt: {both.loc[melty, 'Year']} ({both.loc[melty, 'Bs']} m w.e.)")

---
## Part 3 : adding it up with `.cumsum()`

The bar chart showed single years. To see what the glacier has done **over time**, glaciologists add the years up: the **cumulative mass balance** is a running total of how much mass the glacier has gained or lost since measurements began.

`.cumsum()` (cumulative sum) replaces each value with the sum of everything up to and including it:

In [ ]:
pd.Series([1, -2, 3, -4]).cumsum()

|**value**|1|−2|3|−4|
|:--|:--|:--|:--|:--|
|**running total**|1|1 + (−2) = −1|−1 + 3 = 2|2 + (−4) = −2|

Now the real thing:

In [ ]:
both["cum_Ba"] = both["Ba"].cumsum()                 # NEW

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(both["Year"], both["cum_Ba"], marker="o")
ax.axhline(0, color="black", linewidth=0.8)
ax.set_xlabel("year")
ax.set_ylabel("cumulative balance since 1963 (m w.e.)")
ax.set_title("Rembesdalskåka cumulative mass balance");

print(f"total, 1963-2024: {both['cum_Ba'].iloc[-1]:.2f} m w.e.")
print(f"highest point: {both['cum_Ba'].max():.2f} m w.e. in {both.loc[both['cum_Ba'].idxmax(), 'Year']}")

#### The running total tells a story the bars hid. 
It has three acts:

|**years**|**total change**|**what happened**|
|:--|:--|:--|
|1963–1988|−1.25 m w.e.|roughly in balance|
|1989–1995|+8.22 m w.e.|a run of very snowy winters; the glacier grew|
|1996–2024|−16.43 m w.e.|steady loss, more than wiping out the gains|

Overall, the glacier has lost 9.46 m w.e. since 1963. Ice is a bit less dense than water, so that's roughly a **10-meter-thick** layer of ice taken off the entire glacier.

<div class="alert alert-warning">

**Question:** Almost half the years (29 of 62) were positive. **How can the glacier still have lost almost 10 m w.e.?**

</div>

<details>
<summary style="color:green;"><strong>Answer</strong></summary>

The losing years lost more than the gaining years gained, and the losing years are bunched together recently. A count of good and bad years ignores **how big** each year was; a running total doesn't.

</details>

<div class="alert alert-info">

### Exercise 3 : what did the glacier's front do?

NVE also measures where the glacier's **front** (its downhill edge) is each year: `rembesdalskaka_front_nve.csv`. The `Cumulative front variation (m)` column is how far the front has moved since 1917 (negative = retreat).

1. **Inspect before you read.** Look at the whole file with line numbers: `!cat -n {data}rembesdalskaka_front_nve.csv`. Answer each question below, and next to each question write in this box, the `pd.read_csv` option it decides (or "none needed"):
    - What character separates the columns?
    - Are there any letters that look garbled? What encoding might the file use? (Try `!file -i`.)
    - Do any numbers use a comma as the decimal point?
    - How many header lines are there?
    - Where does the data **end**? What comes after it? How many rows of data are there?
    - Are there any **words** in a column that should be all numbers?
2. Read the file using your answers. Check `.head()`, `tail()`, and `.dtypes()`: `Year` and the cumlative column should both be numbers, and the last row should be 2026.
3. Rename the cumulative column to something short.
4. Make a figure with **2 rows** that share an x-axis: cumulative mass balance on top, cumulative front position on the bottom. Use markers on the bottom panel. Why are markers important for that panel?
5. Did the front respond to the 1989–1995 mass gain?

</div>

<details>
<summary style="color:green;"><strong>Hint 1</strong></summary>

Your answers to step 1 fill in this template. Not every blank is needed, so delete the ones your answers say you don't need:

pd.read_csv(path, encoding=___, sep=___, decimal=___, nrows=___, na_values=___)

To count the data rows, use the line numbers from `cat -n`: the header is line 1, so the number of data rows is (last data line) − 1.

</details>

In [ ]:
# your code here
# 1
!cat -n {data}rembesdalskaka_front_nve.csv

In [ ]:
# 2
front = pd.read_csv("rembesdalskaka_front_nve.csv", ________________________________)

In [ ]:
# 3
front = front.rename(columns={"Cumulative front variation (m)": ________________})
print(front[["Year", "front_m"]].dtypes)

In [ ]:
front.head()

In [ ]:
# 4
fig, ax = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
ax[0].plot(both[______], both[______], marker="o")
ax[0].axhline(0, color="black", linewidth=0.8)
ax[0].set_ylabel("cumulative balance (m w.e.)")

ax[1].plot(front[______], front[______], marker="o")
ax[1].set_ylabel("front position since 1917 (m)")
ax[1].set_xlabel("year");

**Markers matter** because the front wasn't measured every year: there are no measurements between 1921 and `935, or between 1983 and 1995. A line would hide those gaps and suggest the front moved smoothly in between.

---
## Part 4 : saving a summary
At the end of an analysis, you often want a small table of the key numbers, to share, or to combine later with someone else's results (you'll do exactly this in Lab 10).

One way to build a DataFrame from scratch is a **dictionary**: each **key** becomes a column name, each **value** is a **list** with one item per row:

In [ ]:
summary = pd.DataFrame({
    "glacier": ["Rembesdalskåka"],
    "first_year": [both["Year"].min()],
    "last_year": [both["Year"].max()],
    "n_years": [len(both)],
    "n_positive_years": [(both["Ba"] > 0).sum()],
    "cum_balance_mwe": [round(both["cum_Ba"].iloc[-1], 2)],
})

summary

One item in each list means one row. Save it with `.to_csv()` from Lecture 19. `index=False` leaves out the 0, 1, 2, ... index, which would otherwise be written as an extra unnamed column:

In [ ]:
summary.to_csv("rembesdalskaka_summary.csv", index=False)

!cat rembesdalskaka_summary.csv

<div class="alert alert-success">

**Tip:** every list in the dictionary must be the **same length**, because each one is a column of the same table. For two glaciers, each list would have two items: `{"glacier": ["Rembesdalskåka", "Nigardsbreen"], ...}`.

</div>

<div class="alert alert-info">

### Exercise 4 : add to the summary

Add two columns to `summary`: the **mean** annual balance (`mean_Ba`, rounded to 2 decimals), and the total change in front position from 1917 to 2026 (`front_change_m`, from Exercise 3). Save it again and print the file.

</div>

In [ ]:
# your code here

<div style="background-color:#e8daef; border:1px solid #d2b4de; color:#6c3483; padding:10px; border-radius:4px; margin-bottom:20px;">

## Bonus: the equilibrium line

Exercise 1 found the **equilibrium line**: the elevation where a year's net balance is zero. NVE reports it every year in the `Ela` column (meters above sea level), along with `ElaPrefix`. When `ElaPrefix` is `">"`, the equilibrium line was **above the top of the glacier**: there was no part of the glacier that gained mass that year, and the number in `Ela` is just the glacier's top elevation.

1. How many years have `">"`? Were all of them losing years (`Ba < 0`)?
2. Make a scatter plot of `Ela` against `Ba`, using only years with a real equilibrium line (no `">"` and not `NaN`). Describe the relationship, and use `.corr()` to put a number on it.
3. Why would it be wrong to include the `">"` years in that scatter plot as if they were ordinary values?

</div>

In [ ]:
# your code here

<details>
<summary style="color:green;"><strong>Answer</strong></summary>

```python
above = nve[nve["ElaPrefix"] == ">"]
print(len(above), "years with ELA above the glacier;", (above["Ba"] < 0).all())

real = nve[(nve["ElaPrefix"] != ">") & nve["Ela"].notna()]

fig, ax = plt.subplots()
ax.scatter(real["Ela"], real["Ba"])
ax.axhline(0, color="black", linewidth=0.8)
ax.set_xlabel("equilibrium line altitude (m)")
ax.set_ylabel("annual balance (m w.e.)");

print(real["Ela"].corr(real["Ba"]).round(2))
```
1. 19 years have `">"`, and every one of them was a losing year: with no area gaining snow, the glacier can only lose mass.
2. Using the 38 years with a real equilibrium line, the higher the line, the more negative the balance (correlation −0.92). A high equilibrium line means only a small part of the glacier gains snow, while a large part melts.
3. In `">"` years the true equilibrium line was somewhere **above** the glacier, so the `Ela` number is only a lower limit. Plotting it as if it were exact would squash those (very negative) years against the glacier's top elevation, and make the relationship look weaker than it is. Data like this, where you only know a value is "at least" something, is called **censored** data.

</details>

---
## A useful table

|**you want**|**write**|
|:--|:--|
|look at a raw file|`!head -5 path` / `!tail -5 path`|
|read a non-UTF-8 file|`pd.read_csv(path, encoding="latin-1")`|
|columns separated by `;` (or tabs)|`sep=";"` (`sep="\t"`)|
|commas as decimal points|`decimal=","`|
|skip specific lines (e.g. a units row)|`skiprows=[1]`|
|read only the first N rows|`nrows=N`|
|a file with no header row|`header=None, names=["a", "b", ...]`|
|a word that means "missing"|`na_values="Start"`|
|check the column types|`df.dtypes`|
|strip spaces from column names|`df.columns = df.columns.str.strip()`|
|stack arrays|`np.concatenate([a, b], axis=0)`|
|stack DataFrames (matched by name)|`pd.concat([df1, df2], ignore_index=True)`|
|remove exact repeated rows|`df.drop_duplicates()`|
|sort by a column|`df.sort_values("COL", ignore_index=True)`|
|running total|`df["COL"].cumsum()`|
|a DataFrame from scratch|`pd.DataFrame({"col1": [...], "col2": [...]})`|
|save without the index|`df.to_csv("file.csv", index=False)`|